In 20260127_pairwise_to_homeo_weights.ipynb, we found the best weight for each pairwise comparison in respect to homeostatic need. We found a range of weights that can be used for each objective term without penalizing or sacrificing the homeostatic need. Now we want to find the best combination of weights for all terms together. In this notebook, I will be using the epsilon constraint method to find the best combination of weights for all terms together. We have a vague idea of the order of importance of the terms: we value homeostatic objective the most, then kinetics, then efficiency, secretion, and diversity are kind of equal. Secondly, through pareto_exploration.py, we found several sets of weight combinations that can be used without sacrificing the homeostatic need, as an example: lam_sec = 4.07e-10, lam_eff = 8.56e-7, lam_kin = 1.60e-3, lam_div=2.57e-7. We will use these as a prior for our epsilon constraint method.

We will fix the homeostatic term to be within a certain range of the optimal value, and then we will vary the weights of the other terms to find the best combination of weights that maximizes the other terms while still satisfying the homeostatic constraint.

In [2]:
import altair
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp

from ecoli.processes.metabolism_redux_classic import NetworkFlowModel, FlowResult

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

In [3]:
def load_sim(
        time_num: int,
        date: str,
        experiment_name: str,
        condition: str,
        experiment_type: str,
):
    # --- Load Sim ---
    time = str(time_num)
    entry = f'{experiment_name}_{time}_{date}'
    folder_path = f'out/{experiment_type}/{condition}/{entry}/'

    output = np.load(folder_path + '0_output.npy',allow_pickle='TRUE').item()
    output = output['agents']['0']
    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    f = open(folder_path + 'agent_steps.pkl', 'rb')
    agent = dill.load(f)
    f.close()

    metabolism = agent['ecoli-metabolism-redux-classic']

    return fba, bulk, metabolism, output

In [4]:
# import sim
time_num = 600
# date = '2026-01-22'
date = '2026-04-06'
experiment_name = 'homeostatic_only'
condition = 'basal'
experiment_type = 'objective_weight'

fba, bulk, metabolism, output = load_sim(time_num, date, experiment_name, condition, experiment_type)

In [5]:
stoichiometry = metabolism.stoichiometry.copy()
reaction_names = metabolism.reaction_names
metabolites = metabolism.metabolite_names.copy()
counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

homeostatic_only_term = np.array(fba['homeostatic_term'].copy())/counts_to_molar # covert to counts
homeostatic_only_baseline = np.max(homeostatic_only_term) # in counts

homeostatic_dm_targets =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).max(axis=0)
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).mean(axis=0)
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0).mean(axis=0)
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0).mean(axis=0)

In [6]:
homeostatic_dm_targets =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).iloc[1]
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).iloc[1]
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0).iloc[1]
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0).iloc[1]

os.chdir(os.path.expanduser('~/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes'))
from pareto_exploration import run

df = run(
    stoichiometry=stoichiometry,
    metabolites=metabolites,
    reaction_names=reaction_names,
    metabolism=metabolism,
    homeostatic_metabolite_counts=homeostatic_metabolite_counts,
    homeostatic_dm_targets=homeostatic_dm_targets,
    kinetic=kinetic,
    maintenance=maintenance,
    counts_to_molar=counts_to_molar,
    n_samples=10000,
    n_jobs=20,
)

Sampling 10000 weight combinations (log-uniform in feasible ranges)...
Solving 10000 problems (20 parallel job(s))...


  0%|          | 20/10000 [00:00<02:20, 70.83it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
  0%|          | 40/10000 [00:04<20:47,  7.98it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeri

KeyboardInterrupt: 

## Run Pareto Analysis from a Pre-specified CSV of Objective Weights

Instead of random sampling, load a CSV of hand-picked or previously identified weight combinations and solve each one directly, producing the same outputs as the `run()` call above.

**Required CSV columns:** `lambda_sec`, `lambda_eff`, `lambda_kin`, `lambda_div`
**Optional column:** `lambda_hom` (defaults to `1.0` for every row if absent)

In [9]:
import warnings
import polars as pl
from tqdm import tqdm
from joblib import Parallel, delayed
import pareto_exploration as _pe  # already cached in sys.modules from the cell above

os.chdir(os.path.expanduser("~/dev/vEcoli"))  # ensure paths relative to repo root are valid

# ── USER CONFIG ────────────────────────────────────────────────────────────────
CSV_PATH = "notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_10000samples/pareto_results_modified.csv"
OUT_DIR  = "notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_10000samples/modified_from_csv"
N_JOBS   = 20   # increase for parallel solves (same caveat as run() above)
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_DIR, exist_ok=True)
_pe.OUT_DIR = OUT_DIR  # redirect all plot saves to this directory

# --- Load weights from CSV ---
weight_df = pl.read_csv(CSV_PATH)
required_cols = {"lambda_sec", "lambda_eff", "lambda_kin", "lambda_div"}
assert required_cols.issubset(set(weight_df.columns)), (
    f"CSV must contain at minimum: {required_cols}. Found: {set(weight_df.columns)}"
)
if "lambda_hom" not in weight_df.columns:
    weight_df = weight_df.with_columns(pl.lit(1.0).alias("lambda_hom"))
    print("Note: 'lambda_hom' not in CSV — defaulting to 1.0 for all rows.")

weight_samples = weight_df.select(
    ["lambda_hom", "lambda_sec", "lambda_eff", "lambda_kin", "lambda_div"]
).to_numpy()
n_samples = len(weight_samples)
print(f"Loaded {n_samples} weight combinations from '{CSV_PATH}'")

# --- Fixed problem data (set up earlier in this notebook) ---
fixed = dict(
    stoichiometry=stoichiometry,
    metabolites=metabolites,
    reaction_names=reaction_names,
    metabolism=metabolism,
    homeostatic_metabolite_counts=homeostatic_metabolite_counts,
    homeostatic_dm_targets=homeostatic_dm_targets,
    kinetic=kinetic,
    maintenance=maintenance,
    counts_to_molar=counts_to_molar,
)

def _solve_from_csv(i):
    lam_hom, lam_sec, lam_eff, lam_kin, lam_div = weight_samples[i]
    result = _pe.solve_one(lam_hom, lam_sec, lam_eff, lam_kin, lam_div, **fixed)
    if result is not None:
        solution_flux = result.pop("solution_flux")
        base_reaction_flux = metabolism.reaction_mapping_matrix.dot(solution_flux)
        pearson_r2, r2, _ = _pe.correlations_toya_fluxes(
            metabolism.base_reaction_ids, base_reaction_flux
        )
        result["toya_pearson_r_squared"] = pearson_r2
        result["toya_r_squared"] = r2
    return result

# --- Solve ---
print(f"Solving {n_samples} problems ({N_JOBS} parallel job(s))...")
if N_JOBS == 1:
    results = [_solve_from_csv(i) for i in tqdm(range(n_samples))]
else:
    results = Parallel(n_jobs=N_JOBS)(
        delayed(_solve_from_csv)(i) for i in tqdm(range(n_samples))
    )

valid = [r for r in results if r is not None]
n_failed = n_samples - len(valid)
if n_failed:
    warnings.warn(f"{n_failed}/{n_samples} solves failed or were infeasible.")
print(f"  {len(valid)} successful solves.")

# --- Save results and generate plots ---
df_csv = pl.DataFrame(valid)
csv_out = os.path.join(OUT_DIR, "pareto_results.csv")
df_csv.write_csv(csv_out)
print(f"  Saved raw results: {csv_out}")

print("Generating plots...")
_pe.plot_pairwise_altair(df_csv)
_pe.plot_pairwise_altair_weighted(df_csv)
_pe.plot_parallel_coordinates_altair(df_csv)
_pe.plot_3d_plotly(df_csv)
print("Done.")

Loaded 10000 weight combinations from 'notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_10000samples/pareto_results_modified.csv'
Solving 10000 problems (20 parallel job(s))...




  0%|          | 0/10000 [00:00<?, ?it/s]

  0%|          | 20/10000 [00:00<02:03, 80.82it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


  0%|          | 40/10000 [00:05<24:40,  6.73it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/Users/heenasaqib/dev/vEcoli/.venv

  10000 successful solves.
  Saved raw results: notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_10000samples/modified_from_csv/pareto_results.csv
Generating plots...
  Saved: notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_10000samples/modified_from_csv/pairwise_analysis.html
  Saved: notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_10000samples/modified_from_csv/pairwise_analysis_weighted.html
  Saved: notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_10000samples/modified_from_csv/parallel_coordinates.html
  Saved: notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_10000samples/modified_from_csv/pareto_3d.html
Done.
